# 5-Way Attention Comparison — LC25000 / NCT100k

Standalone notebook. Switch datasets via `SELECTED_DATASET` in the dataset-registry cell (`"LC25000"` or `"NCT100k"`, both pre-split with their own official train/test folders). Scope: **MHA-Net, LoRaS-CT, Linformer, Performer, BigBird** — trained
end-to-end on LC25000, plus an isolated attention-mechanism complexity breakdown. Nothing
else (no multi-seed, no K-Fold, no hyperparameter sweep, no foundation models, no
KD-weighting comparison).

**Design**: all 5 attention mechanisms are compared at the **same** `embed_dim`, `num_heads`,
`num_layers` (uniform config) — this isolates "attention mechanism type" as the only
variable, which is the standard practice in the efficient-attention literature (how the
Linformer/Performer papers compare against vanilla attention). LoRaS-CT additionally has its
own `rank` parameter (its defining architectural feature, with no equivalent in the others).

**Output**:
1. `FINAL COMPARISON` table — Test Accuracy, Params, FLOPs, Inference Time, Peak Memory (all 5, trained)
2. `Table 10`-style isolated attention-complexity table — Attn. Params, Attn. FLOPs, Full-Model Params, Full-Model FLOPs, Inference Time, Peak Memory (isolated, architecture-only — doesn't require training)

Includes the established fixes from the main pipeline: patient-safe splitting isn't needed
here (LC25000 ships its own official train/val/test split), the sparse-attention
`masked_fill(-inf)` fix, and per-model checkpointing (so an interruption doesn't require
retraining everything).

## 1. Setup

In [ ]:
!pip install --upgrade pip setuptools wheel --quiet
!pip install numpy pandas matplotlib torch torchvision timm scikit-learn thop tqdm --quiet


In [ ]:
import os
import time
import random
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.multiprocessing as mp
try:
    mp.set_start_method('spawn', force=True)
except RuntimeError:
    pass  # already set (e.g. re-running this cell) -- harmless
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, random_split, Subset
from timm import create_model
from timm.layers import DropPath
from thop import profile
from tqdm.auto import tqdm

random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)
torch.cuda.manual_seed_all(random_seed)

# ============================================================
# GLOBAL CONFIG -- shared UNIFORMLY across all 5 attention types, so the only
# difference between models is the attention mechanism itself. rank is
# LoRaS-CT's own architectural parameter (no equivalent in the other 4).
# ============================================================
GLOBAL_NUM_LAYERS = 2
GLOBAL_NUM_HEADS = 8
GLOBAL_RANK = 32
GLOBAL_NUM_EPOCHS = 10

# ============================================================
# QUICK TEST MODE -- smoke-test the whole 5-model pipeline in a few minutes
# before committing to the full run. Tiny data subsets + fewer epochs.
# ============================================================
QUICK_TEST_MODE = False
QUICK_TEST_MAX_TRAIN = 64
QUICK_TEST_MAX_VAL = 16
QUICK_TEST_MAX_TEST = 16
RUN_EPOCHS = 2 if QUICK_TEST_MODE else GLOBAL_NUM_EPOCHS

from torch.utils.data import Subset as _Subset
def quick_subset(torch_dataset, max_n):
    # NOTE: picks a RANDOM sample of indices (seeded), not the first N --
    # the underlying data is grouped/sorted by class, so taking the first N
    # would silently give a single-class subset and break downstream
    # per-class metrics (classification_report, etc.).
    if not QUICK_TEST_MODE:
        return torch_dataset
    n = min(max_n, len(torch_dataset))
    rng = random.Random(random_seed)
    idx = rng.sample(range(len(torch_dataset)), n)
    return _Subset(torch_dataset, idx)

if QUICK_TEST_MODE:
    print("QUICK_TEST_MODE is ON -- using tiny data subsets / fewer epochs to smoke-test the pipeline.")

assert GLOBAL_RANK % GLOBAL_NUM_HEADS == 0, "GLOBAL_RANK must be divisible by GLOBAL_NUM_HEADS"

# Set True to ignore any saved checkpoints and force every model to retrain from scratch.
FORCE_RETRAIN = False

OUTPUT_ROOT = "/kaggle/working" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else "./outputs"
os.makedirs(OUTPUT_ROOT, exist_ok=True)


## 2. Dataset path

In [ ]:
# NCT100k ships a genuine official train/test split (separate patient slides) -- kept as-is.
# LC25000's "Train and Validation Set" / "Test Set" folders are NOT an official split from
# the dataset authors -- they were split at the image level by whoever packaged this copy,
# which leaks near-duplicate augmentations across train/test (see Section 5a/5b below).
# We still read from these same two paths, but Section 5b pools and re-splits them properly.
DATASET_CONFIGS = {
    "LC25000": {
        "train_val_path": "/scratch/home/admin/lung_colon_image_set/Train and Validation Set",
        "test_path": "/scratch/home/admin/lung_colon_image_set/Test Set",
    },
    "NCT100k": {
        "train_val_path": "/scratch/home/admin/NCT-CRC-HE-100K/",
        "test_path": "/scratch/home/admin/CRC-VAL-HE-7K",
    },
}

SELECTED_DATASET = "LC25000"   # <-- change to "NCT100k" to switch datasets
CURRENT_DATASET = SELECTED_DATASET

TRAIN_VAL_PATH = DATASET_CONFIGS[SELECTED_DATASET]["train_val_path"]
TEST_PATH = DATASET_CONFIGS[SELECTED_DATASET]["test_path"]

assert os.path.isdir(TRAIN_VAL_PATH), f"TRAIN_VAL_PATH not found: {TRAIN_VAL_PATH} -- edit DATASET_CONFIGS above."
assert os.path.isdir(TEST_PATH), f"TEST_PATH not found: {TEST_PATH} -- edit DATASET_CONFIGS above."


In [ ]:
train_val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

batch_size = 32


## 3. Model definitions (5 attention mechanisms, shared CNN backbone)

In [ ]:
class ResNet18_Features(nn.Module):
    def __init__(self):
        super(ResNet18_Features, self).__init__()
        resnet = models.resnet18(pretrained=True)
        self.features = nn.Sequential(*list(resnet.children())[:-2])

    def forward(self, x):
        return self.features(x)  # (B, 512, 7, 7)


class DenseNet121_Features(nn.Module):
    def __init__(self):
        super(DenseNet121_Features, self).__init__()
        densenet = models.densenet121(pretrained=True)
        self.features = densenet.features

    def forward(self, x):
        x = self.features(x)
        x = F.relu(x, inplace=False)
        return x  # (B, 1024, 7, 7)


In [ ]:
class LowRankSparseMultiheadAttention(nn.Module):
    """Attention computed entirely in rank-space (dimension r) -- Q, K, V are
    never projected back up to full embed_dim before the attention product."""
    def __init__(self, embed_dim, num_heads, rank, sparsity_ratio=0.5):
        super(LowRankSparseMultiheadAttention, self).__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        assert rank % num_heads == 0, "rank must be divisible by num_heads (rank-space multi-head split)"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.rank = rank
        self.rank_head_dim = rank // num_heads
        self.sparsity_ratio = sparsity_ratio

        self.q_low = nn.Linear(embed_dim, rank, bias=False)
        self.k_low = nn.Linear(embed_dim, rank, bias=False)
        self.v_low = nn.Linear(embed_dim, rank, bias=False)
        self.rank_mix = nn.Linear(rank, rank, bias=False)
        self.out_proj = nn.Linear(rank, embed_dim, bias=False)
        self.scale = rank ** -0.5

    def sparse_attention(self, attn_scores, sparsity_ratio):
        """Masks non-top-k positions with -inf BEFORE softmax (not by zeroing the
        score and softmaxing over everything) -- zeroing a raw logit and then
        softmaxing still gives that position exp(0)=1 in the numerator, which can
        outweigh genuinely-kept positions whenever their raw scores are negative."""
        batch_size, num_heads, seq_length, _ = attn_scores.size()
        if seq_length == 1:
            return attn_scores
        num_to_keep = max(1, int(sparsity_ratio * seq_length))
        top_scores, _ = torch.topk(attn_scores, k=num_to_keep, dim=-1)
        threshold = top_scores.min(dim=-1, keepdim=True)[0]
        sparse_mask = attn_scores >= threshold
        return attn_scores.masked_fill(~sparse_mask, float('-inf'))

    def forward(self, x):
        batch_size, seq_length, embed_dim = x.size()
        Q = self.q_low(x)
        K = self.k_low(x)
        V = self.v_low(x)
        Q = Q.view(batch_size, seq_length, self.num_heads, self.rank_head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_length, self.num_heads, self.rank_head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_length, self.num_heads, self.rank_head_dim).transpose(1, 2)
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        sparse_attn_scores = self.sparse_attention(attn_scores, self.sparsity_ratio)
        attn_probs = F.softmax(sparse_attn_scores, dim=-1)
        attn_output = torch.matmul(attn_probs, V)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_length, self.rank)
        attn_output = self.rank_mix(attn_output)
        return self.out_proj(attn_output)


class CustomDeiTLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, rank, mlp_ratio=4., drop_path=0.1, sparsity_ratio=0.5):
        super(CustomDeiTLayer, self).__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = LowRankSparseMultiheadAttention(embed_dim, num_heads, rank, sparsity_ratio)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim), nn.GELU(), nn.Linear(mlp_hidden_dim, embed_dim),
        )

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class HybridStudentModel(nn.Module):
    """LoRaS-CT."""
    def __init__(self, num_classes, embed_dim=768, num_heads=GLOBAL_NUM_HEADS, num_layers=GLOBAL_NUM_LAYERS,
                 rank=GLOBAL_RANK, drop_path_rate=0.1, sparsity_ratio=0.5, grid_size=3):
        super(HybridStudentModel, self).__init__()
        self.resnet = ResNet18_Features()
        self.densenet = DenseNet121_Features()
        self.grid_size = grid_size
        concat_channels = 512 + 1024

        self.deit_layers = nn.ModuleList([
            CustomDeiTLayer(embed_dim, num_heads, rank, drop_path=drop_path_rate,
                             sparsity_ratio=sparsity_ratio) for _ in range(num_layers)
        ])
        self.deit_embed = nn.Linear(concat_channels, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        resnet_feats = self.resnet(x)
        densenet_feats = self.densenet(x)
        combined_feats = torch.cat((resnet_feats, densenet_feats), dim=1)
        if self.grid_size != combined_feats.shape[-1]:
            combined_feats = F.adaptive_avg_pool2d(combined_feats, (self.grid_size, self.grid_size))
        b, c, h, w = combined_feats.shape
        combined_feats = combined_feats.view(b, c, h * w).permute(0, 2, 1)
        x = self.deit_embed(combined_feats)
        for layer in self.deit_layers:
            x = layer(x)
        x = self.norm(x)
        pooled = x.mean(dim=1)
        return self.classifier(pooled)


In [ ]:
class CustomDeiTLayer_MHA(nn.Module):
    """Standard multi-head attention, written with explicit nn.Linear ops so thop
    can profile its FLOPs (nn.MultiheadAttention is opaque to thop)."""
    def __init__(self, embed_dim, num_heads, mlp_ratio=4., drop_path=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.scale = self.head_dim ** -0.5
        self.drop_path = nn.Identity() if drop_path == 0 else nn.Dropout(drop_path)
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim), nn.GELU(), nn.Linear(mlp_hidden_dim, embed_dim),
        )

    def forward(self, x):
        normed = self.norm1(x)
        B, N, E = normed.shape
        Q = self.q_proj(normed).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(normed).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(normed).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        attn = F.softmax(torch.matmul(Q, K.transpose(-2, -1)) * self.scale, dim=-1)
        attn_out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, E)
        attn_out = self.out_proj(attn_out)
        x = x + self.drop_path(attn_out)
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class MHANetBaseline(nn.Module):
    """Same spatial CNN backbone as HybridStudentModel, but standard MHA.
    Deliberately uses its OWN fixed num_heads/num_layers (not GLOBAL_NUM_HEADS/
    GLOBAL_NUM_LAYERS) -- those globals get tuned for LoRaS-CT specifically via
    grid search elsewhere in this project, and MHA-Net silently inheriting a
    config tuned for a different architecture would be an unfair/accidental
    comparison. Here, though, both are explicitly built at the SAME GLOBAL_*
    values by construction (see Section 8 below) for a clean attention-only
    comparison -- this class's own defaults are just a safety net."""
    def __init__(self, num_classes, embed_dim=768, num_heads=8, num_layers=2,
                 drop_path_rate=0.1, grid_size=3):
        super().__init__()
        self.resnet = ResNet18_Features()
        self.densenet = DenseNet121_Features()
        concat_channels = 512 + 1024
        self.grid_size = grid_size
        self.deit_layers = nn.ModuleList([
            CustomDeiTLayer_MHA(embed_dim, num_heads, drop_path=drop_path_rate) for _ in range(num_layers)
        ])
        self.deit_embed = nn.Linear(concat_channels, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        resnet_feats = self.resnet(x)
        densenet_feats = self.densenet(x)
        combined_feats = torch.cat((resnet_feats, densenet_feats), dim=1)
        if self.grid_size != combined_feats.shape[-1]:
            combined_feats = F.adaptive_avg_pool2d(combined_feats, (self.grid_size, self.grid_size))
        b, c, h, w = combined_feats.shape
        combined_feats = combined_feats.view(b, c, h * w).permute(0, 2, 1)
        x = self.deit_embed(combined_feats)
        for layer in self.deit_layers:
            x = layer(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        return self.classifier(x)


In [ ]:
# ============================================================
# Efficient-attention baselines (Linformer, Performer, BigBird) + a generic
# CNN-backbone wrapper so any attn_module can be dropped in as a full trainable
# model -- isolates the attention mechanism as the only architectural variable.
# ============================================================
class LinformerAttention(nn.Module):
    """Projects K, V along the sequence dimension N -> k (Wang et al., 2020)."""
    def __init__(self, embed_dim, num_heads, seq_len, proj_k=None):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.seq_len = seq_len
        self.proj_k = proj_k or max(1, seq_len // 2)

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.E_proj = nn.Linear(seq_len, self.proj_k, bias=False)
        self.F_proj = nn.Linear(seq_len, self.proj_k, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.scale = self.head_dim ** -0.5

    def forward(self, x):
        B, N, E = x.shape
        assert N == self.seq_len, f"LinformerAttention was built for N={self.seq_len}, got {N}"
        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.E_proj(self.k_proj(x).transpose(1, 2)).transpose(1, 2)
        V = self.F_proj(self.v_proj(x).transpose(1, 2)).transpose(1, 2)
        K = K.view(B, self.proj_k, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, self.proj_k, self.num_heads, self.head_dim).transpose(1, 2)
        attn = F.softmax(torch.matmul(Q, K.transpose(-2, -1)) * self.scale, dim=-1)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, E)
        return self.out_proj(out)


class PerformerAttention(nn.Module):
    """FAVOR+-style linear attention with a fixed random feature map (Choromanski et al., 2021)."""
    def __init__(self, embed_dim, num_heads, nb_features=None):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.nb_features = nb_features or self.head_dim

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.register_buffer('random_matrix', torch.randn(self.num_heads, self.head_dim, self.nb_features))

    def _phi(self, x):
        proj = torch.einsum('bhnd,hdf->bhnf', x, self.random_matrix)
        return F.elu(proj) + 1

    def forward(self, x):
        B, N, E = x.shape
        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        Qp, Kp = self._phi(Q), self._phi(K)
        KV = torch.einsum('bhnf,bhnd->bhfd', Kp, V)
        denom = torch.einsum('bhnf,bhf->bhn', Qp, Kp.sum(dim=2)) + 1e-6
        out = torch.einsum('bhnf,bhfd->bhnd', Qp, KV) / denom.unsqueeze(-1)
        out = out.transpose(1, 2).contiguous().view(B, N, E)
        return self.out_proj(out)


class BigBirdAttention(nn.Module):
    """Block/local/global/random sparse attention (Zaheer et al., 2020). Falls back
    to full dense attention whenever N <= block_size (always true in this architecture)."""
    def __init__(self, embed_dim, num_heads, block_size=64):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.block_size = block_size

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.scale = self.head_dim ** -0.5

    def forward(self, x):
        B, N, E = x.shape
        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        attn = F.softmax(torch.matmul(Q, K.transpose(-2, -1)) * self.scale, dim=-1)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, E)
        return self.out_proj(out)


class StandardMHAWrapper(nn.Module):
    """Standalone standard-MHA module (thop-profilable), used for the isolated
    attention-only complexity table. Identical computation to CustomDeiTLayer_MHA's
    internal attention."""
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.scale = self.head_dim ** -0.5

    def forward(self, x):
        B, N, E = x.shape
        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        attn = F.softmax(torch.matmul(Q, K.transpose(-2, -1)) * self.scale, dim=-1)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, E)
        return self.out_proj(out)


class GenericAttnLayer(nn.Module):
    """Same transformer-layer shell as CustomDeiTLayer, but takes any attn module."""
    def __init__(self, attn_module, embed_dim, mlp_ratio=4., drop_path=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = attn_module
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(embed_dim, mlp_hidden), nn.GELU(), nn.Linear(mlp_hidden, embed_dim))

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class GenericHybridModel(nn.Module):
    """Same CNN backbone / embed / pooling / classifier as HybridStudentModel and
    MHANetBaseline, so the comparison isolates the attention mechanism only."""
    def __init__(self, num_classes, attn_factory, embed_dim=768, num_layers=GLOBAL_NUM_LAYERS,
                 grid_size=3, drop_path_rate=0.1):
        super().__init__()
        self.resnet = ResNet18_Features()
        self.densenet = DenseNet121_Features()
        self.grid_size = grid_size
        concat_channels = 512 + 1024
        self.layers = nn.ModuleList([
            GenericAttnLayer(attn_factory(), embed_dim, drop_path=drop_path_rate) for _ in range(num_layers)
        ])
        self.deit_embed = nn.Linear(concat_channels, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        r = self.resnet(x)
        d = self.densenet(x)
        feats = torch.cat((r, d), dim=1)
        if self.grid_size != feats.shape[-1]:
            feats = F.adaptive_avg_pool2d(feats, (self.grid_size, self.grid_size))
        b, c, h, w = feats.shape
        feats = feats.view(b, c, h * w).permute(0, 2, 1)
        x = self.deit_embed(feats)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        return self.classifier(x)


class DistillationLoss(nn.Module):
    def __init__(self, alpha=0.5, temperature=3.0):
        super(DistillationLoss, self).__init__()
        self.alpha = alpha
        self.temperature = temperature
        self.ce_loss = nn.CrossEntropyLoss()
        self.kl_div = nn.KLDivLoss(reduction="batchmean")

    def forward(self, student_logits, teacher_logits, ground_truth):
        hard_loss = self.ce_loss(student_logits, ground_truth)
        soft_loss = self.kl_div(
            F.log_softmax(student_logits / self.temperature, dim=1),
            F.softmax(teacher_logits / self.temperature, dim=1)
        ) * (self.temperature ** 2)
        return self.alpha * soft_loss + (1 - self.alpha) * hard_loss


## 4. Shared utilities (train/eval/measure/checkpoint)

In [ ]:
def evaluate_model(model, data_loader, criterion):
    model.eval()
    correct, total, test_loss = 0, 0, 0.0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.cuda(), labels.cuda()
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    avg_loss = test_loss / len(data_loader)
    return accuracy, avg_loss


def train_model_with_distillation(student_model, teacher_models, train_loader, val_loader,
                                   distillation_criterion, optimizer, num_epochs=1, verbose=True):
    for epoch in range(num_epochs):
        student_model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"  epoch {epoch+1}/{num_epochs}", leave=False)
        for images, labels in pbar:
            images, labels = images.cuda(), labels.cuda()
            optimizer.zero_grad()
            student_outputs = student_model(images)
            with torch.no_grad():
                teacher_logits = [teacher(images) for teacher in teacher_models]
                combined_teacher_logits = sum(teacher_logits) / len(teacher_logits)
            loss = distillation_criterion(student_outputs, combined_teacher_logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        avg_loss = running_loss / len(train_loader)
        if verbose:
            val_accuracy, val_loss = evaluate_model(student_model, val_loader, distillation_criterion.ce_loss)
            print(f"  epoch [{epoch+1}/{num_epochs}] train_loss={avg_loss:.4f} "
                  f"val_loss={val_loss:.4f} val_acc={val_accuracy:.2f}%")
        else:
            print(f"  epoch [{epoch+1}/{num_epochs}] train_loss={avg_loss:.4f}")
    return student_model


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def measure_inference_time(model, dummy_input, n_runs=50, device='cuda'):
    model.eval()
    with torch.no_grad():
        for _ in range(5):
            _ = model(dummy_input)
    if device == 'cuda':
        torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        for _ in range(n_runs):
            _ = model(dummy_input)
    if device == 'cuda':
        torch.cuda.synchronize()
    end = time.time()
    return (end - start) / n_runs * 1000  # ms


def measure_memory(model, dummy_input, device='cuda'):
    if device != 'cuda':
        return None
    torch.cuda.reset_peak_memory_stats(device)
    model.eval()
    with torch.no_grad():
        _ = model(dummy_input)
    return torch.cuda.max_memory_allocated(device) / (1024 ** 2)  # MB


def measure_efficiency_isolated(models_dict, dummy_input, device='cuda', n_runs=50, warmup=5):
    """Measures Peak Memory + Inference Time for EVERY model in models_dict, one at a
    time, moving all OTHER models off-GPU first so none can inflate another's peak-memory
    reading. Memory/time depend only on architecture, not trained weights, so freshly-
    constructed (untrained) instances give the same numbers as trained ones."""
    results = {}
    original_devices = {name: next(m.parameters()).device for name, m in models_dict.items()}

    for name, model in models_dict.items():
        for other_name, other_model in models_dict.items():
            if other_name != name:
                other_model.to('cpu')
        torch.cuda.empty_cache()
        gc.collect()

        model.to(device).eval()
        with torch.no_grad():
            for _ in range(warmup):
                _ = model(dummy_input)
        if device == 'cuda':
            torch.cuda.synchronize()

        if device == 'cuda':
            torch.cuda.reset_peak_memory_stats(device)
        with torch.no_grad():
            _ = model(dummy_input)
        if device == 'cuda':
            torch.cuda.synchronize()
        peak_mem = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if device == 'cuda' else None

        if device == 'cuda':
            torch.cuda.synchronize()
        start = time.time()
        with torch.no_grad():
            for _ in range(n_runs):
                _ = model(dummy_input)
        if device == 'cuda':
            torch.cuda.synchronize()
        inf_time = (time.time() - start) / n_runs * 1000

        results[name] = {
            'Inference Time (ms)': round(inf_time, 4),
            'Peak Memory (MB)': round(peak_mem, 4) if peak_mem is not None else None,
        }
        print(f"[{name}] measured in isolation -> Time: {inf_time:.4f} ms, Memory: {peak_mem:.2f} MB")

    for name, model in models_dict.items():
        model.to(original_devices[name])

    return pd.DataFrame(results).T


In [ ]:
# ============================================================
# Per-model checkpointing -- so an interruption doesn't require retraining
# everything. Each model's checkpoint is saved once it finishes its full
# num_epochs run; a re-run loads it back instead of retraining.
# ============================================================
def checkpoint_path(current_dataset, model_name):
    safe_name = model_name.replace(" ", "_").replace("(", "").replace(")", "")
    return f"{OUTPUT_ROOT}/{current_dataset}_{safe_name}_checkpoint.pth"


def save_model_checkpoint(current_dataset, model_name, model, result_dict):
    path = checkpoint_path(current_dataset, model_name)
    clean_state_dict = {k: v for k, v in model.state_dict().items()
                         if not k.endswith(('total_ops', 'total_params'))}
    torch.save({'model_state_dict': clean_state_dict, 'result': result_dict}, path)
    print(f"  [checkpoint saved] {path}")


def load_model_checkpoint(current_dataset, model_name, model):
    path = checkpoint_path(current_dataset, model_name)
    if FORCE_RETRAIN or not os.path.exists(path):
        return None
    ckpt = torch.load(path, map_location='cuda' if torch.cuda.is_available() else 'cpu')
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    print(f"  [checkpoint found] Skipping training for '{model_name}' -- loaded from {path}")
    return ckpt['result']


## 5. Load data + teacher models

### 5a. Group mapping for leakage-safe LC25000 split

In [ ]:
# ============================================================
# Fetch LC25000-clean group mapping (fixes LC25000 augmentation-duplicate
# leakage). Source: https://github.com/GeorgeBatch/LC25000-clean
#
# We only need ONE file from that repo: kaggle/lc25000_image_groups.csv
# (~1.2 MB) -- no need to clone the whole thing.
# ============================================================
import os as _os
import urllib.request as _urlreq

# --- If you already have the CSV somewhere (e.g. via Kaggle "+ Add Data" ->
# --- search "LC25000 Clean Image Groups" / gbatchkala/lc25000-clean-groups,
# --- OR you downloaded it elsewhere and scp'd it over), point this at it.
# --- Leave as None to auto-download from GitHub instead (requires this
# --- notebook's internet to be ON -- it already must be, since the pip
# --- installs in cell 1 need it too).
GROUPS_CSV_OVERRIDE = None   # e.g. "/kaggle/input/lc25000-clean-groups/lc25000_image_groups.csv"

GROUPS_CSV_URL = "https://raw.githubusercontent.com/GeorgeBatch/LC25000-clean/main/kaggle/lc25000_image_groups.csv"
# NOTE: never point the download target at /kaggle/input/... -- that mount is
# READ-ONLY. Downloads always land in a writable spot (OUTPUT_ROOT).
GROUPS_CSV_DOWNLOAD_TARGET = _os.path.join(OUTPUT_ROOT, "lc25000_image_groups.csv")

if GROUPS_CSV_OVERRIDE and _os.path.isfile(GROUPS_CSV_OVERRIDE):
    GROUPS_CSV = GROUPS_CSV_OVERRIDE
elif _os.path.isfile(GROUPS_CSV_DOWNLOAD_TARGET):
    GROUPS_CSV = GROUPS_CSV_DOWNLOAD_TARGET
else:
    GROUPS_CSV = GROUPS_CSV_DOWNLOAD_TARGET
    if SELECTED_DATASET == "LC25000":
        print(f"Downloading group mapping to {GROUPS_CSV} ...")
        try:
            _urlreq.urlretrieve(GROUPS_CSV_URL, GROUPS_CSV)
            print("Download succeeded.")
        except Exception as e:
            print(f"Auto-download failed ({e}).")
            print(
                "Two likely causes on Kaggle:\n"
                "  A) This notebook's internet is OFF. Go to the notebook side panel "
                "-> Settings -> Internet -> toggle ON, then re-run this cell.\n"
                "  B) No internet at all in this environment. Manual fallback:\n"
                "       1. On any machine with internet:\n"
                f"            curl -L -o lc25000_image_groups.csv {GROUPS_CSV_URL}\n"
                "       2. Upload that file as a private Kaggle Dataset (Kaggle -> "
                "Datasets -> New Dataset -> upload the CSV), attach it to this "
                "notebook via '+ Add Data', then set:\n"
                "            GROUPS_CSV_OVERRIDE = \"/kaggle/input/<your-dataset-slug>/lc25000_image_groups.csv\"\n"
                "          above and re-run this cell."
            )

if SELECTED_DATASET == "LC25000":
    assert _os.path.isfile(GROUPS_CSV), (
        f"Group mapping still not found at {GROUPS_CSV}. See instructions printed "
        f"above (or re-run after fixing internet access / manual upload)."
    )
    print(f"Group mapping ready at {GROUPS_CSV}")


### 5b. Build datasets/loaders (group-aware split for LC25000)

In [ ]:
%%writefile lc25000_dataset_utils.py
# Written to disk (not defined inline) because DataLoader workers use
# multiprocessing 'spawn' (see cell 2) -- spawned worker processes need to
# re-IMPORT the Dataset class from an actual module; a class defined in a
# notebook cell lives in __main__ and can't be pickled/found by them,
# causing "Can't get attribute 'PathListDataset' on <module '__main__'>".
import torch
from PIL import Image


class PathListDataset(torch.utils.data.Dataset):
    """Dataset built from an explicit (filepath, label_idx) list -- lets us
    ignore the pre-existing folder split entirely and build our own,
    leakage-safe one."""
    def __init__(self, samples, transform=None):
        self.samples = samples  # list of (path, label_idx)
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)
        return img, label


class ClassesShim:
    """Minimal stand-in for ImageFolder's .classes attribute, since other
    cells (e.g. Section 9's classification_report) reference
    train_val_dataset.classes."""
    def __init__(self, classes):
        self.classes = classes


In [ ]:
from lc25000_dataset_utils import PathListDataset, ClassesShim


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from collections import Counter

if SELECTED_DATASET == "LC25000":
    # ------------------------------------------------------------------
    # LC25000 was built by augmenting ~250 original slide crops per class
    # up to 5,000 images each (~20 near-duplicate rotations/flips per
    # original -- see Borkowski et al. 2019, arXiv:1912.12142). There is NO
    # official train/test split from the dataset authors: the
    # "Train and Validation Set" / "Test Set" folders on disk were split at
    # the IMAGE level, which puts near-duplicate augmentations of the same
    # original tissue crop on both sides of the split -> data leakage ->
    # inflated, near-100% accuracy for every model.
    #
    # Fix: pool ALL images from both folders, discard that split entirely,
    # and re-split at the ORIGINAL-IMAGE ("prototype") level using the group
    # mapping from LC25000-clean
    # (https://github.com/GeorgeBatch/LC25000-clean), so every augmented
    # sibling of a given original stays on the same side of the split.
    # ------------------------------------------------------------------
    groups_df = pd.read_csv(GROUPS_CSV)
    fname_to_group = dict(zip(groups_df["filename"], groups_df["group_id"]))

    class_names = sorted(d.name for d in os.scandir(TRAIN_VAL_PATH) if d.is_dir())
    test_class_names = sorted(d.name for d in os.scandir(TEST_PATH) if d.is_dir())
    assert class_names == test_class_names, (
        f"Class mismatch between train/val and test folders: "
        f"{class_names} vs {test_class_names}."
    )
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    num_classes = len(class_names)
    print(f"Classes ({num_classes}): {class_names}")

    all_paths, all_groups, all_labels = [], [], []
    missing_from_map = 0

    for root in (TRAIN_VAL_PATH, TEST_PATH):
        for cname in class_names:
            class_dir = os.path.join(root, cname)
            if not os.path.isdir(class_dir):
                continue
            for fname in os.listdir(class_dir):
                if not fname.lower().endswith((".jpeg", ".jpg", ".png")):
                    continue
                gid = fname_to_group.get(fname)
                if gid is None:
                    missing_from_map += 1
                    continue  # can't safely place this image without a group id
                all_paths.append(os.path.join(class_dir, fname))
                all_groups.append(gid)
                all_labels.append(class_to_idx[cname])

    print(f"Pooled {len(all_paths)} images "
          f"({missing_from_map} skipped -- filename not found in group mapping; "
          f"investigate before proceeding if this is more than a handful).")

    all_groups = np.array(all_groups)
    all_labels = np.array(all_labels)
    all_paths = np.array(all_paths, dtype=object)

    # Step 1: hold out ~15% of GROUPS (not images) for the test set, stratified by class.
    N_TEST_SPLITS = 6  # ~1/6 ≈ 16.7% held out as test
    sgkf_test = StratifiedGroupKFold(n_splits=N_TEST_SPLITS, shuffle=True, random_state=random_seed)
    trainval_idx, test_idx = next(sgkf_test.split(all_paths, all_labels, groups=all_groups))

    # Step 2: split the remaining groups into train / val (~90/10 of trainval).
    N_VAL_SPLITS = 10
    sgkf_val = StratifiedGroupKFold(n_splits=N_VAL_SPLITS, shuffle=True, random_state=random_seed)
    train_sub_idx, val_sub_idx = next(sgkf_val.split(
        all_paths[trainval_idx], all_labels[trainval_idx], groups=all_groups[trainval_idx]
    ))
    train_idx = trainval_idx[train_sub_idx]
    val_idx = trainval_idx[val_sub_idx]

    # Sanity check: zero group overlap across splits (this is the whole point).
    g_train, g_val, g_test = set(all_groups[train_idx]), set(all_groups[val_idx]), set(all_groups[test_idx])
    assert not (g_train & g_val), "Group leakage between train and val!"
    assert not (g_train & g_test), "Group leakage between train and test!"
    assert not (g_val & g_test), "Group leakage between val and test!"

    print(f"Train: {len(train_idx):>6} imgs / {len(g_train):>4} groups")
    print(f"Val:   {len(val_idx):>6} imgs / {len(g_val):>4} groups")
    print(f"Test:  {len(test_idx):>6} imgs / {len(g_test):>4} groups")
    print("Group overlap across splits: NONE (verified).")
    print("Class balance (train):", Counter(all_labels[train_idx]))
    print("Class balance (val):  ", Counter(all_labels[val_idx]))
    print("Class balance (test): ", Counter(all_labels[test_idx]))

    train_dataset = PathListDataset(
        list(zip(all_paths[train_idx].tolist(), all_labels[train_idx].tolist())),
        transform=train_val_transform,
    )
    val_dataset = PathListDataset(
        list(zip(all_paths[val_idx].tolist(), all_labels[val_idx].tolist())),
        transform=train_val_transform,
    )
    test_dataset_base = PathListDataset(
        list(zip(all_paths[test_idx].tolist(), all_labels[test_idx].tolist())),
        transform=test_transform,
    )

    train_val_dataset = ClassesShim(class_names)  # for .classes used later (Section 9)

else:
    # NCT100k: train and test come from genuinely separate patient slides
    # (no augmentation-duplicate mechanism), so the official split is kept as-is.
    train_val_dataset = datasets.ImageFolder(TRAIN_VAL_PATH)
    test_dataset_base = datasets.ImageFolder(TEST_PATH)
    assert train_val_dataset.classes == test_dataset_base.classes, (
        f"Class mismatch between train/val and test folders: "
        f"{train_val_dataset.classes} vs {test_dataset_base.classes}."
    )
    num_classes = len(train_val_dataset.classes)
    print(f"Classes ({num_classes}): {train_val_dataset.classes}")

    val_size = int(len(train_val_dataset) * 0.1)
    train_size = len(train_val_dataset) - val_size
    train_dataset, val_dataset = random_split(train_val_dataset, [train_size, val_size])
    train_dataset.dataset.transform = train_val_transform
    val_dataset.dataset.transform = train_val_transform
    test_dataset_base.transform = test_transform

train_dataset = quick_subset(train_dataset, QUICK_TEST_MAX_TRAIN)
val_dataset = quick_subset(val_dataset, QUICK_TEST_MAX_VAL)
test_dataset_base = quick_subset(test_dataset_base, QUICK_TEST_MAX_TEST)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, persistent_workers=True)
test_loader = DataLoader(test_dataset_base, batch_size=batch_size, shuffle=False, num_workers=4, persistent_workers=True)

teacher_vit = create_model('vit_base_patch16_224', pretrained=True, num_classes=num_classes).cuda()
teacher_deit = create_model('deit_base_patch16_224', pretrained=True, num_classes=num_classes).cuda()
teacher_swin = create_model('swin_base_patch4_window7_224', pretrained=True, num_classes=num_classes).cuda()
teacher_models = [teacher_vit, teacher_deit, teacher_swin]
for t in teacher_models:
    t.eval()
print("Teacher models loaded.")


## 6. Train all 5 models + FINAL COMPARISON

Each model is trained once, with the shared `GLOBAL_NUM_HEADS`/`GLOBAL_NUM_LAYERS` config
(LoRaS-CT additionally uses `GLOBAL_RANK`).

In [ ]:
N = 3 * 3  # grid_size=3 -> 9 tokens

models_to_compare = {
    'MHA-Net (baseline)': MHANetBaseline(num_classes, num_heads=GLOBAL_NUM_HEADS, num_layers=GLOBAL_NUM_LAYERS,
                                          grid_size=3).cuda(),
    'LoRaS-CT': HybridStudentModel(num_classes, grid_size=3).cuda(),
    'Linformer': GenericHybridModel(
        num_classes, attn_factory=lambda: LinformerAttention(768, GLOBAL_NUM_HEADS, seq_len=N),
        num_layers=GLOBAL_NUM_LAYERS, grid_size=3).cuda(),
    'Performer': GenericHybridModel(
        num_classes, attn_factory=lambda: PerformerAttention(768, GLOBAL_NUM_HEADS),
        num_layers=GLOBAL_NUM_LAYERS, grid_size=3).cuda(),
    'BigBird': GenericHybridModel(
        num_classes, attn_factory=lambda: BigBirdAttention(768, GLOBAL_NUM_HEADS, block_size=64),
        num_layers=GLOBAL_NUM_LAYERS, grid_size=3).cuda(),
}

results = {}
trained_models = {}

for name, model in models_to_compare.items():
    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print(f"{'='*60}")

    cached = load_model_checkpoint(CURRENT_DATASET, name, model)
    if cached is not None:
        results[name] = cached
        trained_models[name] = model
        continue

    criterion = DistillationLoss(alpha=0.5, temperature=3.0)
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    model = train_model_with_distillation(
        model, teacher_models, train_loader, val_loader, criterion, optimizer,
        num_epochs=RUN_EPOCHS, verbose=False
    )

    test_accuracy, test_loss = evaluate_model(model, test_loader, criterion.ce_loss)
    print(f"[{name}] Test Accuracy: {test_accuracy:.2f}%, Test Loss: {test_loss:.4f}")

    dummy_input = torch.randn(1, 3, 224, 224).cuda()
    params = count_parameters(model)
    flops, _ = profile(model, inputs=(dummy_input,), verbose=False)
    inf_time = measure_inference_time(model, dummy_input, device='cuda')
    memory = measure_memory(model, dummy_input, device='cuda')

    results[name] = {
        'Test Accuracy (%)': test_accuracy,
        'Params (M)': params / 1e6,
        'FLOPs (G)': flops / 1e9,
        'Inference Time (ms)': inf_time,
        'Peak Memory (MB)': memory,
    }
    trained_models[name] = model
    save_model_checkpoint(CURRENT_DATASET, name, model, results[name])

    del model
    torch.cuda.empty_cache()
    gc.collect()

print(f"\n{'='*80}")
print(f"FINAL COMPARISON -- {CURRENT_DATASET}")
print(f"{'='*80}")
metrics = ['Test Accuracy (%)', 'Params (M)', 'FLOPs (G)', 'Inference Time (ms)', 'Peak Memory (MB)']
header = f"{'Metric':<25}" + "".join(f"{name:<25}" for name in results.keys())
print(header)
for metric in metrics:
    row = f"{metric:<25}"
    for name in results.keys():
        row += f"{results[name][metric]:<25.4f}"
    print(row)

final_comparison_df = pd.DataFrame(results).T


## 7. Isolated attention-mechanism complexity table

Architecture-only breakdown -- doesn't need training. Same idea as the project's "Table 10":
compares each attention type at the same `embed_dim`/`num_heads`/`num_layers`, in isolation
(no other model resident on GPU), so nothing here is inflated by leftover weights/optimizer
state from a previous measurement.

In [ ]:
embed_dim = 768

def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

attn_factories = {
    'Linformer':               lambda: LinformerAttention(embed_dim, GLOBAL_NUM_HEADS, seq_len=N),
    'Performer':                lambda: PerformerAttention(embed_dim, GLOBAL_NUM_HEADS),
    'BigBird':                  lambda: BigBirdAttention(embed_dim, GLOBAL_NUM_HEADS, block_size=64),
    'Standard MHA (MHA-Net)':   lambda: StandardMHAWrapper(embed_dim, GLOBAL_NUM_HEADS),
    'LoRa-SMHA (proposed)':     lambda: LowRankSparseMultiheadAttention(embed_dim, GLOBAL_NUM_HEADS, rank=GLOBAL_RANK),
}

dummy_img = torch.randn(1, 3, 224, 224)
dummy_seq = torch.randn(1, N, embed_dim)

complexity_rows = {}
isolated_full_models = {}
for name, factory in attn_factories.items():
    attn_module = factory()
    attn_params = count_params(attn_module)
    attn_flops, _ = profile(attn_module, inputs=(dummy_seq,), verbose=False)

    full_model = GenericHybridModel(num_classes, attn_factory=factory, embed_dim=embed_dim,
                                     num_layers=GLOBAL_NUM_LAYERS, grid_size=3)
    full_params = count_params(full_model)
    full_flops, _ = profile(full_model, inputs=(dummy_img,), verbose=False)

    complexity_rows[name] = {
        'Attn. Params': attn_params, 'Attn. FLOPs': attn_flops,
        'Full-Model Params': full_params, 'Full-Model FLOPs': full_flops,
    }
    isolated_full_models[name] = full_model

complexity_table = pd.DataFrame(complexity_rows).T[
    ['Attn. Params', 'Attn. FLOPs', 'Full-Model Params', 'Full-Model FLOPs']
]

print(f"\nTable: isolated attention-mechanism complexity -- {CURRENT_DATASET} "
      f"(embed_dim={embed_dim}, heads={GLOBAL_NUM_HEADS}, layers={GLOBAL_NUM_LAYERS}, rank={GLOBAL_RANK}, N={N})\n")
print(complexity_table.to_string())


In [ ]:
# Isolated Inference Time / Peak Memory for the same 5 full models (freshly built,
# untrained -- architecture determines these, not learned weights).
dummy_img_cuda = torch.randn(1, 3, 224, 224).cuda()
isolated_efficiency_table = measure_efficiency_isolated(isolated_full_models, dummy_img_cuda, device='cuda')

_name_map = {
    'Standard MHA (MHA-Net)': 'MHA-Net (baseline)',
    'LoRa-SMHA (proposed)': 'LoRaS-CT',
    'Linformer': 'Linformer', 'Performer': 'Performer', 'BigBird': 'BigBird',
}
complexity_table['Inference Time (ms)'] = [
    isolated_efficiency_table.loc[n, 'Inference Time (ms)'] for n in complexity_table.index
]
complexity_table['Peak Memory (MB)'] = [
    isolated_efficiency_table.loc[n, 'Peak Memory (MB)'] for n in complexity_table.index
]

print(f"\nTable: isolated attention-mechanism complexity (with Inference Time / Peak Memory) "
      f"-- {CURRENT_DATASET}\n")
print(complexity_table.to_string())

# ------------------------------------------------------------------------
# Overwrite Inference Time / Peak Memory in the Section 6 FINAL COMPARISON
# table with these isolated numbers too. The Section 6 measurements were
# taken on the already-trained model with its optimizer (SGD momentum
# buffers) still resident on GPU, right after a full epoch loop + test-set
# eval + thop.profile() -- reset_peak_memory_stats() only resets the peak
# COUNTER, it does not free memory already allocated by the optimizer, so
# that Peak Memory reading was inflated (a measurement artifact, not a real
# architectural difference). This makes the isolated numbers here the single
# source of truth for both tables -- no separate manual reconciliation needed.
# ------------------------------------------------------------------------
_reverse_name_map = {v: k for k, v in _name_map.items()}
final_comparison_df['Inference Time (ms)'] = [
    isolated_efficiency_table.loc[_reverse_name_map[n], 'Inference Time (ms)'] for n in final_comparison_df.index
]
final_comparison_df['Peak Memory (MB)'] = [
    isolated_efficiency_table.loc[_reverse_name_map[n], 'Peak Memory (MB)'] for n in final_comparison_df.index
]

print(f"\nTable: FINAL COMPARISON, corrected -- {CURRENT_DATASET}\n")
print(final_comparison_df.round(4).to_string())


## 8. Quick visual: accuracy vs. FLOPs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

names = list(final_comparison_df.index)
colors = ['#F18F01' if n == 'MHA-Net (baseline)' else '#2E86AB' if n == 'LoRaS-CT' else '#A9A9A9' for n in names]

axes[0].bar(names, final_comparison_df['Test Accuracy (%)'], color=colors)
axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title(f'{CURRENT_DATASET}: Test Accuracy')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(names, final_comparison_df['FLOPs (G)'], color=colors)
axes[1].set_ylabel('FLOPs (G)')
axes[1].set_title(f'{CURRENT_DATASET}: Full-Model FLOPs')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(f'{OUTPUT_ROOT}/{CURRENT_DATASET}_5way_comparison.png', dpi=150)
plt.show()


In [ ]:
# ============================================================
# Section 9. Precision, Recall, F1-score for all trained models
# ============================================================
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

def get_all_predictions(model, data_loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.cuda(), labels.cuda()
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return all_labels, all_preds

metrics_rows = {}

for name, model in trained_models.items():
    y_true, y_pred = get_all_predictions(model, test_loader)

    precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall_macro    = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1_macro        = f1_score(y_true, y_pred, average='macro', zero_division=0)

    precision_weighted = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall_weighted    = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1_weighted        = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    metrics_rows[name] = {
        'Precision (macro)': precision_macro,
        'Recall (macro)':    recall_macro,
        'F1-score (macro)':  f1_macro,
        'Precision (weighted)': precision_weighted,
        'Recall (weighted)':    recall_weighted,
        'F1-score (weighted)':  f1_weighted,
    }

    print(f"\n{'='*60}")
    print(f"Model: {name}")
    print(f"{'='*60}")
    print(classification_report(
        y_true, y_pred,
        labels=list(range(len(train_val_dataset.classes))),
        target_names=train_val_dataset.classes,
        zero_division=0
    ))

metrics_df = pd.DataFrame(metrics_rows).T
metrics_df = metrics_df.round(4)

print(f"\n{'='*60}")
print(f"FINAL Precision / Recall / F1-score comparison -- {CURRENT_DATASET}")
print(f"{'='*60}\n")
print(metrics_df.to_string())

metrics_df.to_csv(f'{OUTPUT_ROOT}/{CURRENT_DATASET}_precision_recall_f1.csv')
print(f"\n[saved] {OUTPUT_ROOT}/{CURRENT_DATASET}_precision_recall_f1.csv")

In [ ]:
# ============================================================
# Confusion Matrix -- LoRaS-CT
# Paste this in a new cell AFTER Section 6 (once training/loading is done)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

model = trained_models['LoRaS-CT']
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.cuda()
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

class_names = train_val_dataset.classes
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(8, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=True)
ax.set_title(f'Confusion Matrix -- LoRaS-CT ({CURRENT_DATASET})')
plt.tight_layout()
plt.savefig(f'{OUTPUT_ROOT}/{CURRENT_DATASET}_LoRaSCT_confusion_matrix.png', dpi=150)
plt.show()

print(f"\nSaved to {OUTPUT_ROOT}/{CURRENT_DATASET}_LoRaSCT_confusion_matrix.png")